# Modelado MLP (2018–2024)

Este notebook forma parte del pipeline de ciencia de datos del proyecto **crash-severity-predictor**.

El objetivo es desarrollar, entrenar y evaluar un modelo **Multilayer Perceptron (MLP)** para la predicción de severidad en hechos de tránsito a partir del dataset procesado durante las fases de EDA y ETL. Esta implementación constituye la tercera iteración dentro del conjunto de modelos candidatos del proyecto.


In [13]:
# -- Importaciones ----------------------------------------------
import pandas as pd
import numpy as np
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, f1_score, accuracy_score,
                             roc_curve)
import plotly.graph_objects as go
import plotly.express as px
import json, os
import warnings
warnings.filterwarnings('ignore')

print('✓ Librerías cargadas correctamente')

✓ Librerías cargadas correctamente


## 1. Carga de datos

In [14]:
# -- Carga ----------------------------------------------
train = pd.read_parquet('../data/clean/train.parquet')
test  = pd.read_parquet('../data/clean/test.parquet')

FEATURES = ['tipo_eve','tipo_veh','g_hora_5','dia_sem_ocu',
            'sexo_per','edad_quinquenales','mayor_menor','depto_ocu']
TARGET = 'fall_les'

X_train = train[FEATURES]
y_train = train[TARGET]
X_test  = test[FEATURES]
y_test  = test[TARGET]

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Train : {X_train_sc.shape}')
print(f'Test  : {X_test_sc.shape}')
print(f'\n✓ Features escaladas correctamente')

Train : (57553, 8)
Test  : (14389, 8)

✓ Features escaladas correctamente


## 2. Entrenamiento MLP

In [15]:
# -- Entrenamiento con sample_weight para manejar desbalance ----------------------------------------------
sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

mlp = MLPClassifier(
    hidden_layer_sizes=(128, 64, 32),
    activation='relu',
    solver='adam',
    learning_rate_init=0.001,
    max_iter=100,
    early_stopping=True,
    validation_fraction=0.1,
    random_state=42,
    verbose=False
)

mlp.fit(X_train_sc, y_train, sample_weight=sample_weights)
print(f'✓ Modelo entrenado correctamente')
print(f'Capas ocultas  : {mlp.hidden_layer_sizes}')
print(f'Iteraciones    : {mlp.n_iter_}')
print(f'Loss final     : {mlp.loss_:.4f}')

✓ Modelo entrenado correctamente
Capas ocultas  : (128, 64, 32)
Iteraciones    : 27
Loss final     : 0.6052


## 3. Evaluación del modelo

In [16]:
# -- Predicciones ----------------------------------------------
y_pred  = mlp.predict(X_test_sc)
y_proba = mlp.predict_proba(X_test_sc)[:, 1]
auc     = roc_auc_score(y_test, y_proba)
acc     = accuracy_score(y_test, y_pred)
f1      = f1_score(y_test, y_pred, average='weighted')
cm      = confusion_matrix(y_test, y_pred)

print(f'Clases del modelo: {mlp.classes_}')
print(f'\n=== MLP ===')
print(f'Accuracy : {acc:.4f}')
print(f'F1-Score : {f1:.4f}')
print(f'ROC-AUC  : {auc:.4f}')
print(f'\n{classification_report(y_test, y_pred, target_names=["Fallecido","Lesionado"])}')
print(f'Matriz de confusión:')
print(cm)

Clases del modelo: [1 2]

=== MLP ===
Accuracy : 0.6526
F1-Score : 0.6889
ROC-AUC  : 0.7162

              precision    recall  f1-score   support

   Fallecido       0.31      0.68      0.43      2748
   Lesionado       0.89      0.65      0.75     11641

    accuracy                           0.65     14389
   macro avg       0.60      0.66      0.59     14389
weighted avg       0.78      0.65      0.69     14389

Matriz de confusión:
[[1859  889]
 [4110 7531]]


In [17]:
# -- Visualización 1: Métricas ----------------------------------------------
fig_metricas = go.Figure(go.Bar(
    x=['Accuracy', 'F1-Score', 'ROC-AUC'],
    y=[acc, f1, auc],
    text=[f'{acc:.4f}', f'{f1:.4f}', f'{auc:.4f}'],
    textposition='auto',
    marker_color=['#8338EC', '#3A86FF', '#FF006E'],
    width=0.4
))
fig_metricas.update_layout(
    title='Métricas de evaluación — MLP',
    yaxis=dict(range=[0, 1], title='Valor'),
    xaxis_title='Métrica',
    height=400,
    template='plotly_white'
)
fig_metricas.show()

In [18]:
# -- Visualización 2: Matriz de confusión ----------------------------------------------
fig_cm = px.imshow(
    cm,
    labels=dict(x='Predicción', y='Real', color='Cantidad'),
    x=['Fallecido', 'Lesionado'],
    y=['Fallecido', 'Lesionado'],
    text_auto=True,
    color_continuous_scale='Purples',
    title='Matriz de confusión — MLP'
)
fig_cm.update_layout(height=400, template='plotly_white')
fig_cm.show()

In [19]:
# -- Visualización 3: Curva ROC ----------------------------------------------
fpr, tpr, _ = roc_curve(y_test, y_proba, pos_label=2)

fig_roc = go.Figure()
fig_roc.add_trace(go.Scatter(
    x=fpr, y=tpr,
    mode='lines',
    name=f'MLP (AUC = {auc:.4f})',
    line=dict(color='#8338EC', width=2.5)
))
fig_roc.add_trace(go.Scatter(
    x=[0,1], y=[0,1],
    mode='lines',
    name='Baseline (AUC = 0.5)',
    line=dict(color='gray', width=1.5, dash='dash')
))
fig_roc.update_layout(
    title='Curva ROC — MLP',
    xaxis_title='Tasa de Falsos Positivos',
    yaxis_title='Tasa de Verdaderos Positivos',
    height=450,
    template='plotly_white',
    legend=dict(x=0.6, y=0.1)
)
fig_roc.show()

In [20]:
# -- Visualización 4: Curva de pérdida ----------------------------------------------
fig_loss = go.Figure()
fig_loss.add_trace(go.Scatter(
    y=mlp.loss_curve_,
    mode='lines',
    name='Loss entrenamiento',
    line=dict(color='#8338EC', width=2.5)
))
fig_loss.add_trace(go.Scatter(
    y=mlp.validation_scores_,
    mode='lines',
    name='Score validación',
    line=dict(color='#FF006E', width=2.5, dash='dot')
))
fig_loss.update_layout(
    title='Curva de pérdida — MLP',
    xaxis_title='Iteración',
    yaxis_title='Valor',
    height=420,
    template='plotly_white',
    legend=dict(x=0.6, y=0.9)
)
fig_loss.show()

In [21]:
# -- Guardar modelo y scaler ----------------------------------------------
import joblib
import os

os.makedirs('../data/models', exist_ok=True)
joblib.dump(mlp, '../data/models/mlp.pkl')
joblib.dump(scaler, '../data/models/scaler_mlp.pkl')

resultados_mlp = {
    'modelo'             : 'MLP',
    'accuracy'           : round(acc, 4),
    'f1_score'           : round(f1, 4),
    'roc_auc'            : round(auc, 4),
    'precision_fallecido': 0.31,
    'recall_fallecido'   : 0.68,
    'iteraciones'        : mlp.n_iter_,
    'loss_final'         : round(mlp.loss_, 4),
}

with open('../data/models/resultados_mlp.json', 'w') as f:
    json.dump(resultados_mlp, f, indent=2)

print('✓ Modelo guardado en data/models/mlp.pkl')
print('✓ Scaler guardado en data/models/scaler_mlp.pkl')
print(f'\nResumen MLP:')
for k, v in resultados_mlp.items():
    print(f'  {k:<25} {v}')

✓ Modelo guardado en data/models/mlp.pkl
✓ Scaler guardado en data/models/scaler_mlp.pkl

Resumen MLP:
  modelo                    MLP
  accuracy                  0.6526
  f1_score                  0.6889
  roc_auc                   0.7162
  precision_fallecido       0.31
  recall_fallecido          0.68
  iteraciones               27
  loss_final                0.6052


## 4. Resumen del modelo

| Métrica | Valor |
|---|---|
| Accuracy | 65.26% |
| F1-Score (weighted) | 68.89% |
| ROC-AUC | 71.62% |
| Precision Fallecido | 31% |
| Recall Fallecido | 68% |
| Iteraciones | 27 |
| Loss final | 0.6052 |

**Conclusiones:**
- sample_weight="balanced" corrigió el Recall de 9% a 68%
- Convergencia en 27 iteraciones con early stopping
- Loss desciende consistentemente — entrenamiento estable
- Score de validación oscila levemente ; comportamiento normal con desbalance